In [ ]:
# CELL 1: Connection and the same dataset/suite IDs
# Paste your existing private sf_options = {...} connection setup here.
# Reuse the Spark Snowflake connector/approved compute used by the previous notebooks.
# Each notebook is self-contained and can run after a Python restart.
# Dependencies: numpy, pandas, scikit-learn, torch, matplotlib, joblib, plus the Spark Snowflake connector.
# If missing, install these on approved compute before running (no package downloads occur in these cells).
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
if "sf_options" not in globals() or not isinstance(sf_options, dict):
    raise RuntimeError("Paste your existing private sf_options connection dictionary at the top of this cell.")
if "spark" not in globals():
    raise RuntimeError("Run on the Spark/Databricks environment that connected to Snowflake previously.")
sf_options_dl_poc = dict(sf_options)
DATABASE = "DSVC_TAKEDA_TA_PRIVATE"
sf_options_dl_poc.update({"sfDatabase": DATABASE, "sfSchema": "DS_ML"})
SOURCE_PREFIX = "TAK861_TX_READY_V63"
PREFIX = SOURCE_PREFIX + "_DL_POC"
EXPERIMENT_PREFIX = PREFIX + "_BRIAN_SELECTED_V1"
DATASET_ID = "F001"  # Change to freeze a different source snapshot; use the same value in all four notebooks.
SUITE_ID = "S001"    # Change for different settings/code/seeds; use the same value in notebooks 03 and 04.
import re
if any(not re.fullmatch(r"[A-Z][A-Z0-9_]{0,15}", x) for x in (DATASET_ID, SUITE_ID)):
    raise ValueError("Dataset/suite IDs must be short uppercase identifiers.")
PREPARED_TABLE = f"{EXPERIMENT_PREFIX}_{DATASET_ID}_INPUTS"
SPLIT_TABLE = f"{EXPERIMENT_PREFIX}_{DATASET_ID}_SPLIT"
RUN_PREFIX = f"{EXPERIMENT_PREFIX}_{DATASET_ID}_{SUITE_ID}"
SELECTION_TABLE = RUN_PREFIX + "_SELECTION"
EVALUATION_TABLE = RUN_PREFIX + "_EVALUATION"
print("Selected snapshot features; warehouse namespace:", EXPERIMENT_PREFIX)


In [ ]:
# CELL 2: Embedded model loading, metrics and report helpers
"""Validation and preprocessing for the V63 selected snapshot-feature experiment."""
import io
import json
import hashlib
import re
import numpy as np
import pandas as pd


def canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)


def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode()).hexdigest()


def parse_features(value):
    if isinstance(value, str):
        value = json.loads(value)
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, list) or not value or any(not isinstance(x, str) or not x.strip() for x in value):
        raise ValueError("FEATURES must be a nonempty array of exact column names.")
    if len(set(x.upper() for x in value)) != len(value):
        raise ValueError("Duplicate feature names in MODEL_TYPE.FEATURES.")
    prohibited = {"PATIENT_ID", "START_DT", "END_DT", "RESP", "SPLIT", "RND", "SCORE", "DECILE", "CENTILE", "MILLILE"}
    if prohibited.intersection(x.upper() for x in value):
        raise ValueError("The selected list contains an identifier, target, split, random helper or prediction output.")
    return value


def quote_identifier(name):
    return '"' + name.replace('"', '""') + '"'


def normalize_metadata(frame):
    out = frame[["PATIENT_ID", "END_DT", "RESP"]].copy()
    if out.empty or out.isna().any().any():
        raise ValueError("Missing snapshot keys or labels.")
    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x.strip())).all():
        raise ValueError("Patient IDs must remain nonempty strings.")
    dates = pd.to_datetime(out.END_DT, errors="raise")
    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
        raise ValueError("Snapshot cutoffs must be exact dates.")
    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")
    if not out.RESP.isin([0, 1]).all():
        raise ValueError("Nonbinary labels.")
    out["RESP"] = out.RESP.astype("int64")
    if out.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate patient/date keys; no automatic deduplication is permitted.")
    return out


def align_features(snapshots, model_data, features):
    features = parse_features(features)
    expected = normalize_metadata(snapshots).sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    actual = normalize_metadata(model_data)
    missing = set(features).difference(model_data.columns)
    if missing:
        raise ValueError("Selected columns absent from MODEL_DATA: " + repr(sorted(missing)))
    source = actual.copy()
    for name in features:
        # Decimal fractions are converted to float, never through an integer cast.
        source[name] = pd.to_numeric(model_data[name], errors="raise").to_numpy(dtype=np.float64)
    aligned = expected.merge(source, on=["PATIENT_ID", "END_DT"], how="left",
                             validate="one_to_one", suffixes=("", "_SOURCE"), indicator=True)
    if not aligned._merge.eq("both").all() or not aligned.RESP.eq(aligned.RESP_SOURCE).all():
        raise ValueError("Missing source keys or conflicting labels in MODEL_DATA.")
    X = aligned[features].to_numpy(dtype=np.float64)
    if np.isinf(X).any():
        raise ValueError("Infinite source feature values.")
    return expected, X


def raw_hash(X, metadata, features):
    h = hashlib.sha256()
    h.update(canonical_json(features).encode())
    h.update(metadata.to_csv(index=False).encode())
    h.update(canonical_json(list(X.shape)).encode())
    # A separate missing mask avoids platform-dependent NaN payload hashes.
    h.update(np.isnan(X).astype("u1").tobytes())
    h.update(np.nan_to_num(X, nan=0).astype("<f8").tobytes())
    return h.hexdigest()


def npz_bytes(**arrays):
    buffer = io.BytesIO()
    np.savez_compressed(buffer, **arrays)
    return buffer.getvalue()


def json_bytes(value):
    return canonical_json(value).encode()


def bind_split(metadata, frozen, reference):
    original = normalize_metadata(metadata).sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    normalized = normalize_metadata(frozen)
    normalized["SPLIT"] = frozen.SPLIT.to_numpy()
    normalized["SPLIT_CONFIG"] = frozen.SPLIT_CONFIG.to_numpy()
    normalized = normalized.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    if not original.equals(normalized[["PATIENT_ID", "END_DT", "RESP"]]):
        raise ValueError("Saved split differs from the prepared snapshots/labels.")
    if normalized[["SPLIT", "SPLIT_CONFIG"]].isna().any().any():
        raise ValueError("Incomplete frozen split.")
    if set(normalized.SPLIT) != {"train", "validation", "test"}:
        raise ValueError("Unexpected split names.")
    if normalized.groupby("PATIENT_ID").SPLIT.nunique().gt(1).any():
        raise ValueError("Patient leakage between splits.")
    if normalized.SPLIT_CONFIG.nunique() != 1:
        raise ValueError("Inconsistent split configuration.")
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in normalized.itertuples()]
    hashes = {"snapshot_manifest_sha256": digest_json(records),
              "split_config_sha256": digest_json(json.loads(normalized.SPLIT_CONFIG.iloc[0]))}
    if reference.get("run_id") != "RUN_001" or reference.get("training_complete") is not True:
        raise ValueError("Expected completed original RUN_001 reference.")
    if any(reference.get("input_hashes", {}).get(k) != v for k, v in hashes.items()):
        raise ValueError("Patient assignments differ from original RUN_001 fingerprints.")
    for _, part in normalized.groupby("SPLIT"):
        if set(part.RESP) != {0, 1}:
            raise ValueError("Each split needs both outcome classes.")
    return normalized, hashes


def fit_preprocessor(X_train):
    if X_train.ndim != 2 or not len(X_train) or np.isinf(X_train).any():
        raise ValueError("Invalid training feature matrix.")
    all_missing = np.isnan(X_train).all(axis=0)
    median = np.array([0.0 if missing else np.nanmedian(X_train[:, i])
                       for i, missing in enumerate(all_missing)])
    filled = np.where(np.isnan(X_train), median, X_train)
    mean = filled.mean(axis=0)
    scale = filled.std(axis=0)
    scale[scale == 0] = 1.0
    if not np.isfinite(np.r_[median, mean, scale]).all():
        raise ValueError("Nonfinite preprocessing statistics.")
    return {"median": median.tolist(), "mean": mean.tolist(), "scale": scale.tolist(),
            "all_missing_train": all_missing.tolist()}


def transform_features(X, state):
    if X.ndim != 2 or X.shape[1] != len(state["median"]) or np.isinf(X).any():
        raise ValueError("Feature shape or values changed.")
    mask = np.isnan(X)
    values = ((np.where(mask, state["median"], X) - state["mean"]) / state["scale"]).astype(np.float32)
    if not np.isfinite(values).all():
        raise ValueError("Nonfinite standardized values.")
    return values, mask.astype(np.float32)


def rank_tables(metadata, scores):
    labels = metadata.RESP.to_numpy(dtype=int)
    scores = np.asarray(scores, dtype=float)
    if scores.shape != labels.shape or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Invalid evaluation probabilities.")
    ranked = metadata[["PATIENT_ID", "END_DT", "RESP"]].copy()
    ranked["SCORE"] = scores
    ranked = ranked.sort_values(["SCORE", "PATIENT_ID", "END_DT"], ascending=[False, True, True]).reset_index(drop=True)
    n, positives = len(ranked), int(labels.sum())
    base = positives / n
    ranked["DECILE"] = 10 - np.minimum(9, np.arange(n) * 10 // n)
    deciles = []
    cumulative_n = cumulative_positive = 0
    for decile in range(10, 0, -1):
        part = ranked[ranked.DECILE == decile]
        if part.empty:
            continue
        count, positive = len(part), int(part.RESP.sum())
        cumulative_n += count
        cumulative_positive += positive
        rate = positive / count
        deciles.append({"decile": decile, "snapshots": count, "positives": positive,
                        "score_min": float(part.SCORE.min()), "score_max": float(part.SCORE.max()),
                        "response_rate": rate, "lift": rate / base if base else None,
                        "cumulative_snapshots": cumulative_n, "cumulative_positives": cumulative_positive,
                        "cumulative_lift": (cumulative_positive / cumulative_n) / base if base else None,
                        "cumulative_recall": cumulative_positive / positives if positives else None})
    top = []
    for fraction in (.05, .10, .20, .30):
        k = max(1, int(np.ceil(n * fraction)))
        tp = int(ranked.RESP.iloc[:k].sum())
        top.append({"fraction": fraction, "selected": k, "positives": tp, "precision": tp / k,
                    "recall": tp / positives if positives else None, "lift": (tp / k) / base if base else None})
    return pd.DataFrame(deciles), pd.DataFrame(top)


import base64
ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]

def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)

def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows

def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result

def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)

def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")

"""Classification metrics and a threshold selected only on VALIDATION."""

import numpy as np
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)


def _validate(y, probabilities):
    labels = np.asarray(y)
    raw_scores = np.asarray(probabilities)
    if np.iscomplexobj(raw_scores) or (
        raw_scores.dtype == object
        and any(isinstance(value, (complex, np.complexfloating)) for value in raw_scores.flat)
    ):
        raise ValueError("Probabilities must be real values, not complex numbers.")
    scores = np.asarray(raw_scores, dtype=float)
    if labels.ndim != 1 or scores.ndim != 1 or len(labels) != len(scores) or not len(labels):
        raise ValueError("Labels and probabilities must be aligned, nonempty 1-D arrays.")
    if not np.isin(labels, [0, 1]).all():
        raise ValueError("Labels must be binary 0/1.")
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Probabilities must be finite and in [0, 1].")
    return labels.astype(np.int64), scores


def select_validation_threshold(y, probabilities) -> float:
    """Maximize VALIDATION F1; an exact tie uses the highest threshold.

    The caller must provide VALIDATION labels and scores, never TEST. Predictions
    are positive when score >= threshold. Equal scores are never split, and
    integer cross-products identify exact F1 ties without rounding ambiguity.
    """
    labels, scores = _validate(y, probabilities)
    if len(np.unique(labels)) != 2:
        raise ValueError("Threshold selection requires both VALIDATION classes.")
    order = np.argsort(scores, kind="stable")[::-1]
    ranked_scores = scores[order]
    true_positives = np.cumsum(labels[order], dtype=np.int64)
    group_ends = np.r_[np.flatnonzero(ranked_scores[:-1] != ranked_scores[1:]), len(labels) - 1]
    total_positives = int(labels.sum())
    best_numerator, best_denominator = 0, 1
    best_threshold = float(ranked_scores[0])
    for end in group_ends:
        # F1 = 2 TP / (number selected + total positives). Python integers
        # keep cross-products exact and avoid fixed-width integer overflow.
        numerator = 2 * int(true_positives[end])
        denominator = int(end) + 1 + total_positives
        if numerator * best_denominator > best_numerator * denominator:
            best_numerator, best_denominator = numerator, denominator
            best_threshold = float(ranked_scores[end])
        # Descending thresholds retain the highest cutoff on an exact tie.
    return best_threshold


def classification_metrics(y, probabilities, threshold: float) -> dict:
    labels, scores = _validate(y, probabilities)
    if not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("The fixed classification threshold must be in [0, 1].")
    predicted = (scores >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "average_precision": float(average_precision_score(labels, scores)) if labels.sum() else None,
        "roc_auc": float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else None,
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "threshold": float(threshold),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


"""Embedded warehouse orchestration; relies on the embedded core/storage helpers."""

def read_table(table):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load())


def read_query(query):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load())


def configured_features():
    rows = read_table(SOURCE_PREFIX + "_MODEL_TYPE").select("MODEL_TYPE", "FEATURES").collect()
    if len(rows) != 1:
        raise ValueError("Expected exactly one MODEL_TYPE configuration row.")
    features = parse_features(rows[0]["FEATURES"])
    summary = read_table(SOURCE_PREFIX + "_FINAL_MODEL").toPandas()
    if "FEATURES" not in summary.columns:
        raise ValueError("FINAL_MODEL lacks FEATURES.")
    summarized = summary.FEATURES.tolist()
    if any(not isinstance(f, str) or not f for f in summarized):
        raise ValueError("Invalid FINAL_MODEL feature name.")
    comparison = {"configured_model_type": str(rows[0]["MODEL_TYPE"]),
                  "configured_feature_count": len(features), "final_summary_rows": len(summary),
                  "configured_not_in_summary": sorted(set(features) - set(summarized)),
                  "summary_not_in_configuration": sorted(set(summarized) - set(features)),
                  "summary_duplicate_names": sorted(summary.loc[summary.FEATURES.duplicated(), "FEATURES"].unique().tolist()),
                  "rule": "MODEL_TYPE.FEATURES is authoritative, in its stored order; FINAL_MODEL is an audit."}
    columns = set(read_table(SOURCE_PREFIX + "_MODEL_DATA").columns)
    missing = set(["PATIENT_ID", "END_DT", "RESP"] + features).difference(columns)
    if missing:
        raise ValueError("MODEL_DATA is missing required columns: " + repr(sorted(missing)))
    return features, comparison


def fetch_source_features(features):
    fields = ["PATIENT_ID", "END_DT", "RESP"] + features
    selected = ", ".join("M." + quote_identifier(f) for f in fields)
    # Select only the frozen cohort. Duplicated source keys remain visible and fail validation.
    query = (f"SELECT {selected} FROM {DATABASE}.DS_ML.{SOURCE_PREFIX}_MODEL_DATA M "
             f"INNER JOIN (SELECT DISTINCT PATIENT_ID, END_DT FROM {DATABASE}.DS_ML.{PREFIX}_SNAPSHOTS) S "
             "ON M.PATIENT_ID = S.PATIENT_ID AND M.END_DT = S.END_DT")
    return read_query(query).toPandas()


PREPARED_NAMES = {"raw_features.npz", "snapshots.csv", "manifest.json"}
SPLIT_NAMES = {"split.csv", "preprocessor.json", "split_audit.json"}
MODEL_NAMES = {"model.bin", "candidate.json", "history.csv"}
SELECTION_NAMES = {"selection.json", "comparison.csv"}


def population_check(metadata):
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()))
    if observed != (23151, 12447, 1345):
        raise ValueError(f"Frozen V63 cohort changed: snapshots/patients/positives = {observed}")


def prepared_artifacts(metadata, X, features, comparison):
    population_check(metadata)
    manifest = {"schema_version": 1, "dataset_id": DATASET_ID,
                "feature_source": f"{DATABASE}.DS_ML.{SOURCE_PREFIX}_MODEL_DATA",
                "feature_list_source": f"{DATABASE}.DS_ML.{SOURCE_PREFIX}_MODEL_TYPE.FEATURES",
                "features": features, "feature_sha256": digest_json(features),
                "raw_sha256": raw_hash(X, metadata, features), "shape": list(X.shape),
                "min_end_dt": str(metadata.END_DT.min()), "max_end_dt": str(metadata.END_DT.max()),
                "configuration_audit": comparison,
                "representation": "One snapshot row; one numeric value per selected feature. No monthly replication.",
                "historical_feature_availability_verified": False,
                "target": "Existing RESP retained; business objective is 90-day AT escalation; source label construction not independently verified.",
                "evaluation_scope": "Retrospective comparison. Brian's preselected list may have used patients in the current holdout; existing TEST has been inspected."}
    return {"raw_features.npz": npz_bytes(X=X), "snapshots.csv": metadata.to_csv(index=False).encode(),
            "manifest.json": json_bytes(manifest)}


def load_prepared():
    blobs = read_artifacts(PREPARED_TABLE, PREPARED_NAMES)
    manifest = json.loads(blobs["manifest.json"])
    if manifest["dataset_id"] != DATASET_ID or manifest["schema_version"] != 1:
        raise ValueError("Prepared dataset version differs.")
    features = parse_features(manifest["features"])
    metadata = normalize_metadata(pd.read_csv(io.BytesIO(blobs["snapshots.csv"]), dtype={"PATIENT_ID": str}, keep_default_na=False))
    population_check(metadata)
    with np.load(io.BytesIO(blobs["raw_features.npz"]), allow_pickle=False) as arrays:
        X = arrays["X"].copy()
    if list(X.shape) != manifest["shape"] or X.shape != (len(metadata), len(features)):
        raise ValueError("Prepared array shape differs from the manifest.")
    if np.isinf(X).any() or raw_hash(X, metadata, features) != manifest["raw_sha256"]:
        raise ValueError("Prepared inputs fail fingerprint validation.")
    return metadata, X, manifest


def load_experiment():
    metadata, X, manifest = load_prepared()
    blobs = read_artifacts(SPLIT_TABLE, SPLIT_NAMES)
    audit = json.loads(blobs["split_audit.json"])
    preprocessor = json.loads(blobs["preprocessor.json"])
    frozen = pd.read_csv(io.BytesIO(blobs["split.csv"]), dtype={"PATIENT_ID": str}, keep_default_na=False)
    metadata, hashes = bind_split(metadata, frozen, audit["reference_summary"])
    if audit["raw_sha256"] != manifest["raw_sha256"] or hashes != audit["reference_hashes"]:
        raise ValueError("Preparation and split audits refer to different inputs.")
    if digest_json(preprocessor) != audit["preprocessor_sha256"]:
        raise ValueError("Preprocessing settings changed.")
    indices = {k: np.flatnonzero(metadata.SPLIT.to_numpy() == k) for k in ("train", "validation", "test")}
    # Refit only on frozen TRAIN to verify saved settings have no other data dependency.
    if fit_preprocessor(X[indices["train"]]) != preprocessor:
        raise ValueError("Saved preprocessing differs from TRAIN-only fitting.")
    return {"raw_X": X, "metadata": metadata, "manifest": manifest, "preprocessor": preprocessor,
            "audit": audit, "indices": indices, "y": metadata.RESP.to_numpy(dtype=np.int64),
            "input_id": digest_json({"raw": manifest["raw_sha256"], "split": hashes,
                                      "preprocessor": audit["preprocessor_sha256"]})}


def experiment_partition(data, split):
    idx = data["indices"][split]
    X, missing = transform_features(data["raw_X"][idx], data["preprocessor"])
    return X, missing, data["y"][idx]


def split_report(metadata, scores, threshold):
    metrics = classification_metrics(metadata.RESP.to_numpy(), scores, threshold)
    deciles, top = rank_tables(metadata, scores)
    metrics.update({"snapshots": len(metadata), "patients": int(metadata.PATIENT_ID.nunique()),
                    "positives": int(metadata.RESP.sum()), "prevalence": float(metadata.RESP.mean()),
                    "top10_lift": float(top.loc[top.fraction == .10, "lift"].iloc[0])})
    return metrics, deciles, top


def candidate_table(name):
    if not re.fullmatch(r"[a-z][a-z0-9_]*", name):
        raise ValueError("Unexpected recipe name.")
    return f"{RUN_PREFIX}_MODEL_{name.upper()}"


def candidate_contract(data, recipe, settings):
    return {"input_id": data["input_id"], "implementation_sha256": IMPLEMENTATION_SHA256,
            "recipe": recipe, "settings": settings, "suite_id": SUITE_ID, "dataset_id": DATASET_ID}


def plot_reports(reports, history=None):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for split, item in reports.items():
        deciles, top = item["deciles"], item["top"]
        axes[0].plot(deciles.decile, deciles.lift, marker="o", label=split.upper())
        axes[1].plot(top.fraction * 100, top.lift, marker="o", label=split.upper())
        axes[2].plot(top.fraction * 100, top.recall, marker="o", label=split.upper())
    axes[0].set(title="Lift by decile (10 = highest scores)", xlabel="Decile", ylabel="Lift")
    axes[0].invert_xaxis()
    axes[1].set(title="Lift at targeting capacity", xlabel="Top % of snapshots", ylabel="Lift")
    axes[2].set(title="Recall at targeting capacity", xlabel="Top % of snapshots", ylabel="Recall")
    for ax in axes:
        ax.grid(alpha=.25)
        ax.legend()
    fig.tight_layout()
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=150)
    return fig, buffer.getvalue()

IMPLEMENTATION_SHA256 = "fd028ea22e761280d717f2bd1a43404e671d935da0260c5bdf7959106be1efe9"


"""Snapshot-feature models. Fit APIs deliberately accept TRAIN and VALIDATION only."""
import copy
import io
import os
import random
import platform
import time
import joblib
import numpy as np
import torch
from torch import nn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
import sklearn


def default_recipes():
    base = {"width": 64, "heads": 4, "layers": 2, "feedforward": 128,
            "dropout": .20, "weight_decay": .0001, "learning_rate": .0005, "batch_size": 128}
    return [
        {"name": "transformer_baseline", "kind": "transformer", "settings": dict(base)},
        {"name": "transformer_dropout", "kind": "transformer", "settings": dict(base, dropout=.35)},
        {"name": "transformer_decay", "kind": "transformer", "settings": dict(base, weight_decay=.001)},
        {"name": "transformer_small", "kind": "transformer", "settings": dict(base, width=32, layers=1, feedforward=64)},
        {"name": "logistic", "kind": "logistic", "settings": {"C": .1, "max_iter": 2000}},
        {"name": "hist_gradient_boosting", "kind": "hist_gradient_boosting",
         "settings": {"max_iter": 200, "learning_rate": .05, "max_leaf_nodes": 15,
                      "min_samples_leaf": 40, "l2_regularization": 1.0}},
    ]


def seed_selected(seed):
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def selection_key(y, probabilities):
    y, p = np.asarray(y), np.asarray(probabilities)
    if y.ndim != 1 or y.shape != p.shape or set(np.unique(y)) != {0, 1}:
        raise ValueError("Selection requires aligned binary labels with both classes.")
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Invalid selection probabilities.")
    # Input rows have canonical patient/date order. Stable sorting breaks ties by that order.
    order = np.argsort(-p, kind="stable")
    k = max(1, int(np.ceil(.10 * len(y))))
    lift = float(y[order[:k]].mean() / y.mean())
    return lift, float(average_precision_score(y, p))


class FeatureTokenTransformer(nn.Module):
    def __init__(self, features, settings):
        super().__init__()
        width = settings["width"]
        self.features = features
        # Each token has a learned feature identity plus its standardized numeric value.
        self.value_weight = nn.Parameter(torch.randn(features, width) * .02)
        self.feature_bias = nn.Parameter(torch.randn(features, width) * .02)
        self.missing_embedding = nn.Parameter(torch.randn(features, width) * .02)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, width))
        layer = nn.TransformerEncoderLayer(d_model=width, nhead=settings["heads"],
                    dim_feedforward=settings["feedforward"], dropout=settings["dropout"],
                    batch_first=True, activation="gelu", norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=settings["layers"], enable_nested_tensor=False)
        self.output = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 1))

    def forward(self, values, missing):
        if values.ndim != 2 or values.shape[1] != self.features or values.shape != missing.shape:
            raise ValueError("Expected one value and missingness flag per selected feature.")
        tokens = values.unsqueeze(-1) * self.value_weight + self.feature_bias
        tokens = tokens + missing.unsqueeze(-1) * self.missing_embedding
        sequence = torch.cat([self.cls_token.expand(len(values), -1, -1), tokens], dim=1)
        return self.output(self.encoder(sequence)[:, 0]).squeeze(-1)


def network_scores(network, X, missing, device, batch_size=512):
    network.eval()
    scores = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            values = torch.as_tensor(X[start:start+batch_size], dtype=torch.float32, device=device)
            mask = torch.as_tensor(missing[start:start+batch_size], dtype=torch.float32, device=device)
            scores.append(torch.sigmoid(network(values, mask)).cpu().numpy())
    return np.concatenate(scores).astype(float)


def weighted_probability_loss(y, p, positive_weight):
    p = np.clip(p, 1e-7, 1-1e-7)
    return float(np.mean(-positive_weight * y * np.log(p) - (1-y) * np.log1p(-p)))


def fit_candidate(recipe, X_train, missing_train, y_train, X_val, missing_val, y_val,
                  seed=42, max_epochs=30, patience=6):
    for X, mask, y in ((X_train, missing_train, y_train), (X_val, missing_val, y_val)):
        if X.ndim != 2 or X.shape != mask.shape or len(X) != len(y) or set(np.unique(y)) != {0, 1}:
            raise ValueError("Invalid partition dimensions or binary labels.")
        if not np.isfinite(X).all() or not np.isin(mask, [0, 1]).all():
            raise ValueError("Invalid preprocessed values or missingness flags.")
    if X_train.shape[1] != X_val.shape[1] or max_epochs < 1 or patience < 1:
        raise ValueError("Invalid model dimensions or training limits.")
    seed_selected(seed)
    started = time.time()
    positive_weight = float((len(y_train)-y_train.sum()) / y_train.sum())
    result = {"recipe": copy.deepcopy(recipe), "n_features": X_train.shape[1], "seed": seed,
              "history": [], "positive_weight": positive_weight,
              "runtime": {"python": platform.python_version(), "numpy": np.__version__,
                          "sklearn": sklearn.__version__, "torch": str(torch.__version__)}}
    settings = recipe["settings"]
    if recipe["kind"] == "transformer":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        result["runtime"]["device"] = str(device)
        network = FeatureTokenTransformer(X_train.shape[1], settings).to(device)
        optimizer = torch.optim.AdamW(network.parameters(), lr=settings["learning_rate"], weight_decay=settings["weight_decay"])
        loss_function = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=device))
        dataset = torch.utils.data.TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                       torch.tensor(missing_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
        loader = torch.utils.data.DataLoader(dataset, batch_size=settings["batch_size"], shuffle=True,
                       num_workers=0, generator=torch.Generator().manual_seed(seed))
        best_key, best_state, stale = (-np.inf, -np.inf), None, 0
        for epoch in range(1, max_epochs+1):
            network.train()
            loss_sum = 0.0
            for values, mask, labels in loader:
                values, mask, labels = values.to(device), mask.to(device), labels.to(device)
                optimizer.zero_grad(set_to_none=True)
                loss = loss_function(network(values, mask), labels)
                if not torch.isfinite(loss):
                    raise ValueError("Training loss became nonfinite.")
                loss.backward()
                nn.utils.clip_grad_norm_(network.parameters(), 1.0)
                optimizer.step()
                loss_sum += float(loss.detach().cpu()) * len(labels)
            train_scores = network_scores(network, X_train, missing_train, device)
            val_scores = network_scores(network, X_val, missing_val, device)
            train_key, val_key = selection_key(y_train, train_scores), selection_key(y_val, val_scores)
            result["history"].append({"epoch": epoch, "train_batch_loss": loss_sum/len(y_train),
                 "train_loss": weighted_probability_loss(y_train, train_scores, positive_weight),
                 "validation_loss": weighted_probability_loss(y_val, val_scores, positive_weight),
                 "train_top10_lift": train_key[0], "train_ap": train_key[1],
                 "validation_top10_lift": val_key[0], "validation_ap": val_key[1]})
            print(f"{recipe['name']} epoch {epoch}: TRAIN lift={train_key[0]:.3f}; VALIDATION lift={val_key[0]:.3f}, AP={val_key[1]:.4f}", flush=True)
            if val_key > best_key:
                best_key = val_key
                best_state = {k: v.detach().cpu().clone() for k, v in network.state_dict().items()}
                result["best_epoch"] = epoch
                stale = 0
            else:
                stale += 1
                if stale >= patience:
                    break
        result["state_dict"] = best_state
        del network, optimizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        # Missing flags describe selected values; they are not additional warehouse feature sources.
        train_matrix = np.concatenate([X_train, missing_train], axis=1)
        if recipe["kind"] == "logistic":
            estimator = LogisticRegression(**settings, class_weight="balanced", random_state=seed)
            estimator.fit(train_matrix, y_train)
        elif recipe["kind"] == "hist_gradient_boosting":
            estimator = HistGradientBoostingClassifier(**settings, early_stopping=False, random_state=seed)
            estimator.fit(train_matrix, y_train, sample_weight=np.where(y_train == 1, positive_weight, 1.0))
        else:
            raise ValueError("Unknown model family.")
        result["estimator"] = estimator
        result["best_epoch"] = None
        train_scores = estimator.predict_proba(train_matrix)[:, 1]
        val_scores = estimator.predict_proba(np.concatenate([X_val, missing_val], axis=1))[:, 1]
        train_key, val_key = selection_key(y_train, train_scores), selection_key(y_val, val_scores)
        result["history"] = [{"epoch": 0, "train_loss": weighted_probability_loss(y_train, train_scores, positive_weight),
             "validation_loss": weighted_probability_loss(y_val, val_scores, positive_weight),
             "train_top10_lift": train_key[0], "train_ap": train_key[1],
             "validation_top10_lift": val_key[0], "validation_ap": val_key[1]}]
        print(f"{recipe['name']}: TRAIN lift={train_key[0]:.3f}; VALIDATION lift={val_key[0]:.3f}, AP={val_key[1]:.4f}", flush=True)
    result["seconds"] = time.time()-started
    return result


def predict_candidate(result, X, missing):
    if X.ndim != 2 or X.shape != missing.shape or X.shape[1] != result["n_features"]:
        raise ValueError("Prediction feature dimensions changed.")
    if not np.isfinite(X).all() or not np.isin(missing, [0, 1]).all():
        raise ValueError("Invalid prediction inputs.")
    if result["recipe"]["kind"] == "transformer":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        network = FeatureTokenTransformer(result["n_features"], result["recipe"]["settings"]).to(device)
        network.load_state_dict(result["state_dict"])
        p = network_scores(network, X, missing, device)
        del network
    else:
        p = result["estimator"].predict_proba(np.concatenate([X, missing], axis=1))[:, 1]
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Invalid predicted probabilities.")
    return p


def dump_candidate(result):
    buffer = io.BytesIO()
    joblib.dump(result, buffer, compress=3)
    return buffer.getvalue()


def load_candidate(blob):
    # Only load this suite's own checksum-verified, provenance-checked warehouse artifacts.
    # Joblib is pickle based: never use this function for an external/untrusted model file.
    return joblib.load(io.BytesIO(blob))


In [ ]:
# CELL 3: Load the frozen winner and verify provenance
data = load_experiment()
selection_blobs = read_artifacts(SELECTION_TABLE, SELECTION_NAMES)
selection = json.loads(selection_blobs["selection.json"])
if selection["input_id"] != data["input_id"] or selection["contract"]["implementation_sha256"] != IMPLEMENTATION_SHA256:
    raise ValueError("Winner uses different data or code. Run the matching notebook versions.")
if selection["test_used_for_selection"] is not False:
    raise ValueError("Selection must exclude TEST.")
if selection["model_table"] != candidate_table(selection["winner"]):
    raise ValueError("Unexpected model artifact source.")
saved = read_artifacts(selection["model_table"], MODEL_NAMES)
candidate = json.loads(saved["candidate.json"])
expected_contract = candidate_contract(data, selection["recipe"], selection["contract"]["settings"])
if candidate["contract"] != expected_contract or candidate["threshold"] != selection["threshold"]:
    raise ValueError("Saved candidate differs from the frozen winner/threshold.")
model = load_candidate(saved["model.bin"])
threshold = selection["threshold"]
print("Winner:", selection["winner"], "| Frozen validation threshold:", threshold)


In [ ]:
# CELL 4: Score TRAIN, VALIDATION and TEST using the identical saved model
reports, probabilities = {}, {}
for split in ("train", "validation", "test"):
    values, missing, labels = experiment_partition(data, split)
    scores = predict_candidate(model, values, missing)
    part = data["metadata"].iloc[data["indices"][split]].reset_index(drop=True)
    metrics, deciles, top = split_report(part, scores, threshold)
    reports[split] = {"metrics": metrics, "deciles": deciles, "top": top}
    probabilities[split] = scores
if not np.isclose(reports["validation"]["metrics"]["top10_lift"], selection["validation_metrics"]["top10_lift"]):
    raise ValueError("Reloaded model does not reproduce validation lift.")


In [ ]:
# CELL 5: Display classification metrics, full deciles and capacity lift
metrics_frame = pd.DataFrame([dict(split=split, **item["metrics"]) for split, item in reports.items()])
display(metrics_frame)
for split, item in reports.items():
    print(split.upper(), "(in-sample fit diagnostic)" if split == "train" else "")
    display(item["deciles"])
    display(item["top"])
print("Decile 10 contains the highest scores. Ties break by patient/date, never by outcome.")
print("Lift = selected response rate / response rate of that same evaluation split.")


In [ ]:
# CELL 6: Plot lift, recall, precision-recall and confusion matrices
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, ConfusionMatrixDisplay
fig, lift_png = plot_reports(reports)
plt.show()
fig2, axes = plt.subplots(1, 2, figsize=(12, 4))
for split, scores in probabilities.items():
    labels = data["y"][data["indices"][split]]
    precision, recall, _ = precision_recall_curve(labels, scores)
    axes[0].plot(recall, precision, label=split.upper())
axes[0].set(title="Precision-recall by split", xlabel="Recall", ylabel="Precision")
axes[0].legend()
test_metrics = reports["test"]["metrics"]
ConfusionMatrixDisplay(np.array([[test_metrics["tn"], test_metrics["fp"]],
                                [test_metrics["fn"], test_metrics["tp"]]])).plot(ax=axes[1], colorbar=False)
axes[1].set_title("TEST at frozen validation threshold")
fig2.tight_layout()
chart_buffer = io.BytesIO()
fig2.savefig(chart_buffer, format="png", dpi=150)
classification_png = chart_buffer.getvalue()
plt.show()


In [ ]:
# CELL 7: Save reproducible metrics, plots and ranked split scores
report = {"winner": selection["winner"], "input_id": data["input_id"],
          "threshold": threshold, "features": data["manifest"]["features"],
          "implementation_sha256": IMPLEMENTATION_SHA256,
          "metrics": {k: v["metrics"] for k, v in reports.items()},
          "training_lift_is_in_sample": True,
          "selection_rule": selection["contract"]["settings"]["selection"],
          "limitations": ["Existing TEST was already inspected in earlier work.",
            "Selected feature list comes from the client model; its original selection population is unverified.",
            "Historical availability of source features and 90-day label construction are not independently verified.",
            "Snapshot metrics include correlated snapshots within each patient.",
            "A later untouched cohort is needed for independent confirmation of an improvement."]}
artifacts = {"evaluation.json": json_bytes(report), "metrics.csv": metrics_frame.to_csv(index=False).encode(),
             "lift.png": lift_png, "classification.png": classification_png,
             "candidate_comparison.csv": selection_blobs["comparison.csv"]}
for split, item in reports.items():
    artifacts[split + "_deciles.csv"] = item["deciles"].to_csv(index=False).encode()
    artifacts[split + "_top_k.csv"] = item["top"].to_csv(index=False).encode()
# Row-level scores remain in the same private warehouse artifact table; no external export.
score_rows = data["metadata"][["PATIENT_ID", "END_DT", "RESP", "SPLIT"]].copy()
score_rows["SCORE"] = np.nan
for split, scores in probabilities.items():
    score_rows.loc[data["indices"][split], "SCORE"] = scores
artifacts["predictions.csv"] = score_rows.to_csv(index=False).encode()
if table_exists(EVALUATION_TABLE):
    previous = read_artifacts(EVALUATION_TABLE, artifacts.keys())
    if previous["evaluation.json"] != artifacts["evaluation.json"]:
        raise ValueError("Existing evaluation differs. Do not overwrite; investigate the changed run.")
    print("Existing matching evaluation retained (rendered image bytes can differ by runtime).")
else:
    save_artifacts(EVALUATION_TABLE, artifacts)


In [ ]:
# CELL 8: Report training lift, validation lift and retrospective TEST lift
for split in ("train", "validation", "test"):
    m = reports[split]["metrics"]
    print(f"{split.upper()}: top-10% lift={m['top10_lift']:.3f}, AP={m['average_precision']:.4f}, AUC={m['roc_auc']:.4f}")
print("Training lift measures fit to training data; higher training lift alone is not success.")
print("Compare experiments on validation. TEST results are retrospective, not untouched confirmation.")
print("Goal: improve lift; no improvement is claimed until these notebooks run on the private data.")
print("Artifacts saved under:", EVALUATION_TABLE)
